In [ ]:
!pip install dowhy -q
!pip install econml -q
!apt-get update
!apt-get install -y graphviz
!pip install graphviz pydot
!pip install --upgrade dowhy



In [ ]:
from copy import deepcopy
import json
import time
import pydot


import numpy as np
import pandas as pd
from scipy import stats

from sklearn.metrics import mean_absolute_percentage_error, accuracy_score, f1_score
from sklearn.model_selection import train_test_split

import dowhy
from dowhy import CausalModel

from econml.metalearners import SLearner, XLearner, TLearner
from econml.dml import LinearDML, CausalForestDML, DML
from econml.dr import DRLearner, SparseLinearDRLearner

from sklearn.linear_model import LinearRegression, LogisticRegression, LassoCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV

from lightgbm import LGBMRegressor, LGBMClassifier

from networkx.drawing.nx_pydot import from_pydot

import networkx as nx

from tqdm import tqdm

import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')

import graphviz

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMRegressor
from dowhy import CausalModel

In [ ]:
dowhy.__version__

In [ ]:
COLORS = [
    '#00B0F0',
    '#FF0000',
    '#B0F000'
]

In [ ]:
def plot_effect(effect_true, effect_pred, figsize=(10, 7), ylim=(5000, 22000)):
    plt.figure(figsize=figsize)
    plt.scatter(effect_true, effect_pred, color=COLORS[0])
    plt.plot(np.sort(effect_true), np.sort(effect_true), color=COLORS[1], alpha=.7, label='Perfect model')
    plt.xlabel('$True\ effect$', fontsize=14, alpha=.5)
    plt.ylabel('$Predicted\ effect$', fontsize=14, alpha=.5)
    plt.ylim(ylim[0], ylim[1])
    plt.legend()
    plt.show()

### Read the data

In [ ]:
data = pd.read_csv("SEM_scaled_clean_data.csv")

train_data, val_data = train_test_split(
    data,
    test_size=0.3,        # 30% validation, 70% training
    random_state=0,      # ensures reproducibility
    shuffle=True
)

### Define the graph

In [ ]:
import networkx as nx

# Create DAG
G = nx.DiGraph()

# --------------------------
# 1. Nodes
# --------------------------
all_nodes = [
    "select", "empl_06", "dwomen",
    "age_lb", "salary_04", "empl_04",
    "city_1_0","city_2_0","city_3_0","city_4_0","city_5_0","city_6_0",
    "educ_lb_0_0","educ_lb_2_0","educ_lb_3_0","educ_lb_4_0","educ_lb_5_0",
    "educ_lb_6_0","educ_lb_7_0","educ_lb_8_0","educ_lb_9_0","educ_lb_10_0",
    "educ_lb_12_0","educ_lb_13_0","educ_lb_14_0","educ_lb_16_0"
]

G.add_nodes_from(all_nodes)

# --------------------------
# 2. Baseline covariate list
# --------------------------
baseline_covs = [
    "age_lb","salary_04","empl_04",
    "city_1_0","city_2_0","city_3_0","city_4_0","city_5_0","city_6_0",
    "educ_lb_0_0","educ_lb_2_0","educ_lb_3_0","educ_lb_4_0","educ_lb_5_0",
    "educ_lb_6_0","educ_lb_7_0","educ_lb_8_0","educ_lb_9_0","educ_lb_10_0",
    "educ_lb_12_0","educ_lb_13_0","educ_lb_14_0","educ_lb_16_0"
]

# --------------------------
# 3. Edges: confounders
# --------------------------

# Gender is a confounder
G.add_edge("dwomen", "select")
G.add_edge("dwomen", "empl_06")

# Baseline covariates are confounders
for cov in baseline_covs:
    G.add_edge(cov, "select")     # confounder path 1
    G.add_edge(cov, "empl_06")    # confounder path 2

# Optional: gender affects covariates
for cov in baseline_covs:
    G.add_edge("dwomen", cov)

# Age → salary, Age → prior employment
G.add_edge("age_lb", "salary_04")
G.add_edge("age_lb", "empl_04")

# Treatment → outcome
G.add_edge("select", "empl_06")


In [ ]:
# Instantiate the CausalModel 
model = CausalModel(
    data=train_data,
    treatment='select',
    outcome='empl_06',
    graph=G,
    effect_modifiers=['dwomen']  # or [] if you don't need modifiers
)

In [ ]:
model.view_model()

### Get the estimand

In [ ]:
estimand = model.identify_effect()
print(estimand)

# Extract the backdoor (adjustment) variables implied by the DAG
adjustment_set = list(estimand.get_backdoor_variables())
print("Adjustment set:", adjustment_set)

In [ ]:
X = train_data[adjustment_set]     # all DAG-chosen covariates
T = train_data["select"]           # binary treatment
Y = train_data["empl_06"]          # binary outcome (employment at 6 months)
gender = train_data["dwomen"]

In [ ]:
from econml.dr import DRLearner
from sklearn.linear_model import LogisticRegression, LinearRegression
from lightgbm import LGBMRegressor
from sklearn.model_selection import KFold

rng = 123

dr = DRLearner(
    model_regression=LGBMRegressor(
        n_estimators=800,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=rng
    ),
    model_propensity=LogisticRegression(max_iter=2000),
    model_final=LinearRegression(),
    cv=KFold(n_splits=3, shuffle=True, random_state=rng)
)

dr.fit(
    Y,
    T,
    X=X,
    W=None
)

#do a refutation test####


In [ ]:
ate = dr.ate(X)
print("ATE:", ate)



In [ ]:
def fast_bootstrap_ate(dr, X, n_boot=500, seed=123):
    rng = np.random.default_rng(seed)

    # This is the influence-function estimate
    cate = dr.effect(X)
    n = len(cate)
    ates = []

    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        ates.append(cate[idx].mean())

    return np.percentile(ates, 2.5), np.percentile(ates, 97.5)

lb, ub = fast_bootstrap_ate(dr, X, n_boot=500)
print("ATE:", dr.ate(X))
print("95% CI:", (lb, ub))

In [ ]:
cate = dr.effect(X)

# Gender splits
women_mask = train_data["dwomen"] == 1
men_mask   = train_data["dwomen"] == 0

cate_women = cate[women_mask].mean()
cate_men   = cate[men_mask].mean()
diff      = cate_women - cate_men

print("CATE for women:", cate_women)
print("CATE for men:", cate_men)
print("Difference (Women - Men):", diff)

print("Std of CATE (women):", cate[women_mask].std())
print("Std of CATE (men):", cate[men_mask].std())

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="paper", font_scale=1.25)

# ======================================================
# 1. SUMMARY TABLES
# ======================================================

def summarize_ate(ate, lb, ub):
    df = pd.DataFrame({
        "ATE Estimate": [ate],
        "95% CI Lower": [lb],
        "95% CI Upper": [ub],
        "CI Width": [ub - lb]
    })
    return df

def summarize_gender_heterogeneity(cate, women_mask, men_mask):
    ate_women = cate[women_mask].mean()
    ate_men   = cate[men_mask].mean()
    diff      = ate_women - ate_men
    
    df = pd.DataFrame({
        "Group": ["Women", "Men", "Difference (Women - Men)"],
        "ATE": [ate_women, ate_men, diff],
        "Std Dev": [
            cate[women_mask].std(),
            cate[men_mask].std(),
            np.nan
        ],
        "Count": [
            women_mask.sum(),
            men_mask.sum(),
            np.nan
        ]
    })
    return df

# ======================================================
# 2. HISTOGRAM + DENSITY PLOTS
# ======================================================

# ---- FULL CATE DISTRIBUTION ----
def plot_cate_hist(cate):
    plt.figure(figsize=(10,6))
    
    sns.histplot(cate, bins=35, kde=True, color="steelblue", alpha=0.6)
    plt.axvline(cate.mean(), color="red", linestyle="--", linewidth=2, 
                label=f"Mean CATE = {cate.mean():.4f}")
    
    plt.title("Histogram of Individual Treatment Effects (CATE)")
    plt.xlabel("CATE")
    plt.ylabel("Frequency")
    plt.legend()
    plt.show()

# ---- WOMEN VS MEN HISTOGRAM ----
def plot_gender_hist(cate, women_mask, men_mask):
    plt.figure(figsize=(10,6))
    
    sns.histplot(cate[women_mask], bins=30, kde=True, color="purple", 
                 alpha=0.5, label="Women")
    sns.histplot(cate[men_mask], bins=30, kde=True, color="green", 
                 alpha=0.5, label="Men")
    
    plt.axvline(cate[women_mask].mean(), color="purple", linestyle="--",
                linewidth=2, label=f"Women Mean = {cate[women_mask].mean():.4f}")
    plt.axvline(cate[men_mask].mean(), color="green", linestyle="--",
                linewidth=2, label=f"Men Mean = {cate[men_mask].mean():.4f}")
    
    plt.title("Histogram of CATE by Gender")
    plt.xlabel("CATE")
    plt.ylabel("Frequency")
    plt.legend()
    plt.show()

# ---- HISTOGRAM VERSION OF BOX PLOT ----
def plot_gender_hist_separate(cate, women_mask, men_mask):
    plt.figure(figsize=(10,6))
    
    sns.histplot(cate[women_mask], bins=30, kde=True, color="purple", alpha=0.6)
    sns.histplot(cate[men_mask], bins=30, kde=True, color="green", alpha=0.6)
    
    plt.title("Gender Comparison: Overlaid Histograms of CATE")
    plt.xlabel("CATE")
    plt.ylabel("Frequency")
    plt.legend(["Women", "Men"])
    plt.show()

# ======================================================
# 3. RUN EVERYTHING
# ======================================================

# ATE summary
ate_summary = summarize_ate(ate, lb, ub)
print("\n===== ATE SUMMARY =====")
display(ate_summary)

# Gender heterogeneity summary
gender_summary = summarize_gender_heterogeneity(cate, women_mask, men_mask)
print("\n===== GENDER HETEROGENEITY SUMMARY =====")
display(gender_summary)

# Histogram: full CATE
plot_cate_hist(cate)

# Histogram: women vs men
plot_gender_hist(cate, women_mask, men_mask)

# Histogram: gender overlaid
plot_gender_hist_separate(cate, women_mask, men_mask)


In [ ]:
# ------------------------------------------
# 1. Extract inner list of propensity models
# ------------------------------------------
flat_prop_models = dr.models_propensity[0]  
# This contains: [LogisticRegression(), LogisticRegression(), LogisticRegression()]

print("Number of propensity models:", len(flat_prop_models))

# ------------------------------------------
# 2. Compute averaged propensity scores
# ------------------------------------------
propensity_scores = np.mean(
    [m.predict_proba(X)[:, 1] for m in flat_prop_models],
    axis=0
)

# ------------------------------------------
# 3. Build dataframe
# ------------------------------------------
df_ps = pd.DataFrame({
    "propensity": propensity_scores,
    "treatment": T.values
})

df_ps.head()

plt.figure(figsize=(10,6))

# Histogram for treated
sns.histplot(
    df_ps[df_ps["treatment"]==1]["propensity"],
    bins=30,
    kde=False,
    color="blue",
    alpha=0.45,
    label="Treated"
)

# Histogram for control
sns.histplot(
    df_ps[df_ps["treatment"]==0]["propensity"],
    bins=30,
    kde=False,
    color="orange",
    alpha=0.45,
    label="Control"
)

# Median vertical lines
plt.axvline(
    df_ps[df_ps["treatment"]==1]["propensity"].median(),
    color="blue",
    linestyle="--",
    linewidth=1.5
)
plt.axvline(
    df_ps[df_ps["treatment"]==0]["propensity"].median(),
    color="orange",
    linestyle="--",
    linewidth=1.5
)

plt.title("Propensity Score Distribution by Treatment Group (Histogram Overlap)")
plt.xlabel("Estimated Propensity Score P(T=1 | X)")
plt.ylabel("Count")
plt.legend()
plt.show()


In [ ]:
plt.figure(figsize=(12,3))

sns.rugplot(
    df_ps[df_ps["treatment"]==1]["propensity"],
    height=0.2,
    color="blue",
    label="Treated"
)
sns.rugplot(
    df_ps[df_ps["treatment"]==0]["propensity"],
    height=0.2,
    color="orange",
    label="Control"
)

plt.title("Rug Plot — Propensity Score Support by Treatment Status")
plt.xlabel("Propensity Score")
plt.yticks([])
plt.legend()
plt.show()


In [ ]:
plt.figure(figsize=(12,6))

# Control group (bottom)
sns.histplot(
    df_ps[df_ps["treatment"]==0]["propensity"],
    bins=30,
    color="orange",
    alpha=0.6,
    label="Control"
)

# Treated group mirrored (top)
sns.histplot(
    df_ps[df_ps["treatment"]==1]["propensity"],
    bins=30,
    color="blue",
    alpha=0.6,
    label="Treated"
)

plt.title("Mirror Histogram — Positivity Diagnostics")
plt.xlabel("Propensity Score")
plt.legend()
plt.show()


In [ ]:
plt.figure(figsize=(10,6))

sns.ecdfplot(
    df_ps[df_ps["treatment"]==1]["propensity"],
    color="blue",
    label="Treated"
)

sns.ecdfplot(
    df_ps[df_ps["treatment"]==0]["propensity"],
    color="orange",
    label="Control"
)

plt.title("ECDF Plot — Positivity Check")
plt.xlabel("Propensity Score")
plt.ylabel("ECDF")
plt.legend()
plt.show()


In [ ]:
df_ps["ps_quintile"] = pd.qcut(df_ps["propensity"], 5, labels=False)

plt.figure(figsize=(12,6))
sns.countplot(
    data=df_ps,
    x="ps_quintile",
    hue="treatment",
    palette=["orange", "blue"]
)

plt.title("Count by Propensity Score Quintile (Stratification Plot)")
plt.xlabel("Propensity Score Quintile")
plt.ylabel("Count")
plt.legend(["Control","Treated"])
plt.show()


In [ ]:
plt.figure(figsize=(10,6))

sns.scatterplot(
    x=df_ps["propensity"],
    y=train_data["empl_06"],
    hue=df_ps["treatment"],
    palette=["orange", "blue"],
    alpha=0.6
)

plt.title("Scatterplot: Outcome vs Propensity Score")
plt.xlabel("Propensity Score")
plt.ylabel("Outcome (empl_06)")
plt.legend(["Control","Treated"])
plt.show()


In [ ]:
import numpy as np
import pandas as pd

def compute_smd(X, T, weights=None):
    smd = {}
    for col in X.columns:
        x1 = X.loc[T==1, col]
        x0 = X.loc[T==0, col]

        if weights is None:
            w1 = np.ones_like(x1)
            w0 = np.ones_like(x0)
        else:
            w1 = weights.loc[T==1]
            w0 = weights.loc[T==0]

        # Weighted means/SD
        mean1 = np.average(x1, weights=w1)
        mean0 = np.average(x0, weights=w0)
        sd1 = np.sqrt(np.average((x1-mean1)**2, weights=w1))
        sd0 = np.sqrt(np.average((x0-mean0)**2, weights=w0))

        pooled_sd = np.sqrt((sd1**2 + sd0**2) / 2)

        smd[col] = (mean1 - mean0) / pooled_sd

    return pd.Series(smd)
smd_before = compute_smd(X, T)
# Stabilized IPW weights
w = (T / propensity_scores) + ((1 - T) / (1 - propensity_scores))
w = pd.Series(w, index=X.index)

smd_after = compute_smd(X, T, weights=w)


In [ ]:
balance_table = pd.DataFrame({
    "SMD Before": smd_before,
    "SMD After": smd_after
})

display(balance_table)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, len(X.columns)*0.35))

y_pos = np.arange(len(X.columns))

plt.scatter(smd_before.values, y_pos, color="red", label="Before Weighting")
plt.scatter(smd_after.values, y_pos, color="blue", label="After Weighting")

plt.axvline(0, color="black", linewidth=1)
plt.axvline(0.1, color="gray", linestyle="--")
plt.axvline(-0.1, color="gray", linestyle="--")

plt.yticks(y_pos, X.columns)
plt.xlabel("Standardized Mean Difference")
plt.title("Love Plot: Covariate Balance Before and After Weighting")
plt.legend()
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
import seaborn as sns

for col in X.columns:
    plt.figure(figsize=(8,4))
    sns.kdeplot(X.loc[T==1, col], fill=True, color="blue", alpha=0.4, label="Treated")
    sns.kdeplot(X.loc[T==0, col], fill=True, color="orange", alpha=0.4, label="Control")
    plt.title(f"Covariate Density: {col}")
    plt.legend()
    plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

# --- Simple plot function ---
def plot_effect(effect_pred):
    """Plot the distribution of predicted treatment effects (CATEs)."""
    plt.figure(figsize=(7,5))
    sns.histplot(effect_pred, bins=30, kde=True, color="skyblue")
    plt.title("Distribution of Predicted Treatment Effects (CATEs)")
    plt.xlabel("Estimated Effect on Employment (ΔY)")
    plt.ylabel("Frequency")
    plt.show()

# --- Plot your results ---
plot_effect(effect_pred)


In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt
    
    # Plot treatment effect distribution by gender
plt.figure(figsize=(7, 5))
plt.hist(effect_pred[X_test['dwomen'] == 0], bins=30, alpha=0.6, label='Men')
plt.hist(effect_pred[X_test['dwomen'] == 1], bins=30, alpha=0.6, label='Women')
plt.title("Distribution of Estimated Treatment Effects (CATEs)")
plt.xlabel("Estimated Effect on Employment (ΔY)")
plt.ylabel("Frequency")
plt.legend()
plt.show()

In [ ]:
Y = train_data["empl_06"]
T = train_data["select"]

e = propensity_scores  # from your DRLearner propensity model

import numpy as np

# avoid division issues at extreme PS
eps = 1e-6
e_clipped = np.clip(e, eps, 1 - eps)

# weights for treated and control
w_treated  = T / e_clipped
w_control  = (1 - T) / (1 - e_clipped)

mu1_ipw = np.sum(w_treated * Y) / np.sum(w_treated)
mu0_ipw = np.sum(w_control * Y) / np.sum(w_control)

ate_ipw = mu1_ipw - mu0_ipw
print("IPW ATE:", ate_ipw)

# ATT: focus on treated group, reweight controls
w_att_ctrl = (e_clipped / (1 - e_clipped)) * (1 - T)  # controls weighted to look like treated

mu1_att = np.mean(Y[T == 1])  # observed outcome for treated
mu0_att = np.sum(w_att_ctrl * Y) / np.sum(w_att_ctrl)

att_ipw = mu1_att - mu0_att
print("IPW ATT:", att_ipw)

def bootstrap_ipw_ate(Y, T, e, n_boot=200, seed=123):
    rng = np.random.default_rng(seed)
    n = len(Y)
    ates = []

    eps = 1e-6
    e_clip = np.clip(e, eps, 1-eps)

    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        Yb, Tb, eb = Y.iloc[idx], T.iloc[idx], e_clip[idx]

        w_t = Tb / eb
        w_c = (1 - Tb) / (1 - eb)

        mu1 = np.sum(w_t * Yb) / np.sum(w_t)
        mu0 = np.sum(w_c * Yb) / np.sum(w_c)
        ates.append(mu1 - mu0)

    lower = np.percentile(ates, 2.5)
    upper = np.percentile(ates, 97.5)
    return np.mean(ates), lower, upper

ate_ipw_bs, lb_ipw, ub_ipw = bootstrap_ipw_ate(Y, T, e)
print("Bootstrap IPW ATE:", ate_ipw_bs)
print("95% CI:", (lb_ipw, ub_ipw))



In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import KFold
from sklearn.linear_model import LogisticRegression, LinearRegression
from lightgbm import LGBMRegressor
from econml.dr import DRLearner

# ================================================================
# 1. BOOTSTRAP FUNCTIONS
# ================================================================

# ---- IPW Bootstrap ----
def bootstrap_ipw_ate(Y, T, e, n_boot=300, seed=123):
    rng = np.random.default_rng(seed)
    n = len(Y)
    ates = []

    # avoid division at 0 or 1
    eps = 1e-6
    e = np.clip(e, eps, 1 - eps)

    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        Yb, Tb, eb = Y.iloc[idx], T.iloc[idx], e[idx]

        w_t = Tb / eb
        w_c = (1 - Tb) / (1 - eb)

        mu1 = np.sum(w_t * Yb) / np.sum(w_t)
        mu0 = np.sum(w_c * Yb) / np.sum(w_c)

        ates.append(mu1 - mu0)

    return np.array(ates)


# ---- DR Bootstrap (rebuilds model each iteration) ----
def bootstrap_dr_ate(dr_params, X, Y, T, n_boot=300, seed=123):
    rng = np.random.default_rng(seed)
    n = len(Y)
    ates = []

    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        Xb, Yb, Tb = X.iloc[idx], Y.iloc[idx], T.iloc[idx]

        # fresh model each time
        dr_tmp = DRLearner(**dr_params)
        dr_tmp.fit(Yb, Tb, X=Xb)

        ates.append(dr_tmp.ate(Xb))

    return np.array(ates)


# ================================================================
# 2. DEFINE DRLearner PARAMETERS (matches your model)
# ================================================================

rng = 123

dr_params = {
    "model_regression": LGBMRegressor(
        n_estimators=800,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=rng
    ),
    "model_propensity": LogisticRegression(max_iter=2000),
    "model_final": LinearRegression(),
    "cv": KFold(n_splits=3, shuffle=True, random_state=rng)
}


# ================================================================
# 3. RUN BOTH BOOTSTRAPS
# ================================================================

print("Bootstrapping IPW ATE...")
ipw_boot = bootstrap_ipw_ate(Y, T, propensity_scores, n_boot=300)

print("Bootstrapping DR ATE...")
dr_boot = bootstrap_dr_ate(dr_params, X, Y, T, n_boot=300)


# ================================================================
# 4. SUMMARY TABLE
# ================================================================

summary_df = pd.DataFrame({
    "Estimator": ["IPW", "DR"],
    "Mean ATE": [ipw_boot.mean(), dr_boot.mean()],
    "Lower 95%": [np.percentile(ipw_boot, 2.5), np.percentile(dr_boot, 2.5)],
    "Upper 95%": [np.percentile(ipw_boot, 97.5), np.percentile(dr_boot, 97.5)],
})

print("\n================= ATE Bootstrap Summary =================")
print(summary_df)


# ================================================================
# 5. BOOTSTRAP DISTRIBUTION PLOT
# ================================================================

plt.figure(figsize=(12, 6))

sns.kdeplot(ipw_boot, fill=True, color="red", alpha=0.4, label="IPW Bootstrap ATE")
sns.kdeplot(dr_boot, fill=True, color="blue", alpha=0.4, label="DR Bootstrap ATE")

# Means
plt.axvline(ipw_boot.mean(), color="red", linestyle="--", label="IPW Mean")
plt.axvline(dr_boot.mean(), color="blue", linestyle="--", label="DR Mean")

# 95% CI lines
plt.axvline(np.percentile(ipw_boot, 2.5), color="red", linestyle=":")
plt.axvline(np.percentile(ipw_boot, 97.5), color="red", linestyle=":")
plt.axvline(np.percentile(dr_boot, 2.5), color="blue", linestyle=":")
plt.axvline(np.percentile(dr_boot, 97.5), color="blue", linestyle=":")

plt.title("Bootstrap Distribution of ATE: IPW vs DRLearner", fontsize=16)
plt.xlabel("ATE Estimate")
plt.ylabel("Density")
plt.grid(alpha=0.3)
plt.legend()
plt.show()


In [ ]:
from econml.metalearners import SLearner, TLearner, XLearner
from lightgbm import LGBMRegressor
from sklearn.linear_model import LogisticRegression

base_reg = LGBMRegressor(
    n_estimators=800,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=123
)

propensity_learner = LogisticRegression(max_iter=2000)

# ----------------- S-Learner -----------------
s_learner = SLearner(overall_model=base_reg)
s_learner.fit(Y, T, X=X)

# ----------------- T-Learner -----------------
# single model will be cloned for treated & control internally
t_learner = TLearner(models=base_reg)
t_learner.fit(Y, T, X=X)

# ----------------- X-Learner -----------------
x_learner = XLearner(
    models=base_reg,
    propensity_model=propensity_learner
)

x_learner.fit(Y, T, X=X)


In [ ]:
gender = train_data["dwomen"]

# ----------------- CATE for each learner -----------------
cate_S = s_learner.effect(X)          # effect(X, T0=0, T1=1) defaults to 0→1
cate_T = t_learner.effect(X)
cate_X = x_learner.effect(X)

# ----------------- Overall ATE -----------------
ate_S = cate_S.mean()
ate_T = cate_T.mean()
ate_X = cate_X.mean()

print("S-Learner ATE:", ate_S)
print("T-Learner ATE:", ate_T)
print("X-Learner ATE:", ate_X)

# ----------------- Gender-specific effects -----------------
mask_w = (gender == 1)
mask_m = (gender == 0)

def subgroup_summary(cate, name):
    ate_w = cate[mask_w].mean()
    ate_m = cate[mask_m].mean()
    diff  = ate_w - ate_m
    print(f"\n=== {name} ===")
    print("ATE (Women):", ate_w)
    print("ATE (Men):  ", ate_m)
    print("Diff (W - M):", diff)

subgroup_summary(cate_S, "S-Learner")
subgroup_summary(cate_T, "T-Learner")
subgroup_summary(cate_X, "X-Learner")


In [ ]:
summary = pd.DataFrame({
    "Estimator": ["S-Learner", "T-Learner", "X-Learner"],
    "ATE": [ate_S, ate_T, ate_X],
    "ATE (Women)": [cate_S[mask_w].mean(), cate_T[mask_w].mean(), cate_X[mask_w].mean()],
    "ATE (Men)":   [cate_S[mask_m].mean(), cate_T[mask_m].mean(), cate_X[mask_m].mean()]
})
summary


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))
sns.kdeplot(cate_S, fill=True, alpha=0.4, label="S-Learner")
sns.kdeplot(cate_T, fill=True, alpha=0.4, label="T-Learner")
sns.kdeplot(cate_X, fill=True, alpha=0.4, label="X-Learner")

plt.title("CATE Distribution: S vs T vs X Learner", fontsize=15)
plt.xlabel("Estimated Treatment Effect")
plt.ylabel("Density")
plt.grid(alpha=0.3)
plt.legend()
plt.show()


In [ ]:
plt.figure(figsize=(12,6))

sns.kdeplot(cate_S[mask_w], fill=True, color="purple", alpha=0.3, label="S-Learner (Women)")
sns.kdeplot(cate_S[mask_m], fill=True, color="green",  alpha=0.3, label="S-Learner (Men)")

sns.kdeplot(cate_T[mask_w], fill=False, color="purple", linestyle="--", label="T-Learner (Women)")
sns.kdeplot(cate_T[mask_m], fill=False, color="green",  linestyle="--", label="T-Learner (Men)")

sns.kdeplot(cate_X[mask_w], fill=False, color="purple", linestyle=":", label="X-Learner (Women)")
sns.kdeplot(cate_X[mask_m], fill=False, color="green",  linestyle=":", label="X-Learner (Men)")

plt.title("Gender-Specific CATE Distributions\n(Women vs Men)", fontsize=15)
plt.xlabel("Estimated Treatment Effect")
plt.ylabel("Density")
plt.grid(alpha=0.3)
plt.legend()
plt.show()


In [ ]:
df_cate = pd.DataFrame({
    "CATE_S": cate_S,
    "CATE_T": cate_T,
    "CATE_X": cate_X,
    "Gender": gender.replace({1: "Women", 0: "Men"})
})

plt.figure(figsize=(12,6))
sns.boxplot(data=df_cate, x="Gender", y="CATE_S")
plt.title("S-Learner: CATE by Gender")
plt.show()

plt.figure(figsize=(12,6))
sns.boxplot(data=df_cate, x="Gender", y="CATE_T")
plt.title("T-Learner: CATE by Gender")
plt.show()

plt.figure(figsize=(12,6))
sns.boxplot(data=df_cate, x="Gender", y="CATE_X")
plt.title("X-Learner: CATE by Gender")
plt.show()


In [ ]:
plt.figure(figsize=(10,6))

sns.stripplot(
    x=df_cate["Gender"],
    y=cate_X, 
    jitter=0.3, 
    alpha=0.5, 
    color="blue"
)

plt.title("X-Learner: CATE vs Gender (Jittered Points)")
plt.ylabel("Estimated Treatment Effect")
plt.xlabel("Gender")
plt.grid(alpha=0.3)
plt.show()


In [ ]:
df_long = pd.DataFrame({
    "CATE": np.concatenate([cate_S, cate_T, cate_X]),
    "Learner": (["S-Learner"]*len(cate_S) +
                ["T-Learner"]*len(cate_T) +
                ["X-Learner"]*len(cate_X)),
    "Gender": pd.concat([gender, gender, gender]).replace({1:"Women", 0:"Men"}).values
})

plt.figure(figsize=(12,6))
sns.boxplot(data=df_long, x="Learner", y="CATE", hue="Gender")
plt.title("CATE Distributions by Learner and Gender")
plt.grid(alpha=0.3)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# -------------------------------------------------------------------
# Prepare tidy dataframe
# -------------------------------------------------------------------
df = pd.DataFrame({
    "S-Learner": cate_S,
    "T-Learner": cate_T,
    "X-Learner": cate_X,
    "Gender": gender.replace({1: "Women", 0: "Men"})
})

df_long = df.melt(
    id_vars="Gender",
    value_vars=["S-Learner", "T-Learner", "X-Learner"],
    var_name="Learner",
    value_name="CATE"
)

# Setup gender subsets
df_women = df_long[df_long["Gender"] == "Women"]
df_men   = df_long[df_long["Gender"] == "Men"]

# -------------------------------------------------------------------
# Publication-ready style
# -------------------------------------------------------------------
plt.style.use("default")
sns.set_theme(context="paper", style="whitegrid", font_scale=1.3)

fig = plt.figure(figsize=(16, 12))

# ============================================================
# Panel A: Overall CATE distribution
# ============================================================
ax1 = fig.add_subplot(2, 2, 1)

sns.kdeplot(
    data=df_long[df_long["Learner"] == "S-Learner"],
    x="CATE", fill=True, alpha=0.35, linewidth=1.6, label="S-Learner"
)
sns.kdeplot(
    data=df_long[df_long["Learner"] == "T-Learner"],
    x="CATE", fill=True, alpha=0.35, linewidth=1.6, label="T-Learner"
)
sns.kdeplot(
    data=df_long[df_long["Learner"] == "X-Learner"],
    x="CATE", fill=True, alpha=0.35, linewidth=1.6, label="X-Learner"
)

ax1.set_title("Panel A: Distribution of Estimated CATE")
ax1.set_xlabel("Estimated Treatment Effect")
ax1.set_ylabel("Density")
ax1.legend(frameon=True)
ax1.grid(alpha=0.3)

# ============================================================
# Panel B: Gender-specific CATE distributions (fixed)
# ============================================================
ax2 = fig.add_subplot(2, 2, 2)

# Women (solid)
sns.kdeplot(
    data=df_women, x="CATE", hue="Learner",
    fill=True, alpha=0.25, linewidth=1.6, ax=ax2
)

# Men (dashed)
sns.kdeplot(
    data=df_men, x="CATE", hue="Learner",
    fill=False, linewidth=1.6, linestyle="--", ax=ax2,
    legend=False
)

ax2.set_title("Panel B: CATE by Gender")
ax2.set_xlabel("Estimated Treatment Effect")
ax2.set_ylabel("Density")
ax2.grid(alpha=0.3)

handles, labels = ax2.get_legend_handles_labels()
ax2.legend(handles, labels, title="Learner", frameon=True)

# ============================================================
# Panel C: Boxplots by Learner × Gender
# ============================================================
ax3 = fig.add_subplot(2, 1, 2)

sns.boxplot(
    data=df_long, x="Learner", y="CATE", hue="Gender",
    palette="Set2", linewidth=1.2, fliersize=2.5, ax=ax3
)

ax3.set_title("Panel C: CATE by Learner and Gender")
ax3.set_xlabel("Learner Method")
ax3.set_ylabel("Estimated Treatment Effect")
ax3.grid(alpha=0.3)
ax3.legend(title="Gender", frameon=True)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# -------------------------------------------------------------------
# Prepare tidy dataframe
# -------------------------------------------------------------------
df = pd.DataFrame({
    "S-Learner": cate_S,
    "T-Learner": cate_T,
    "X-Learner": cate_X,
    "Gender": gender.replace({1: "Women", 0: "Men"})
})

df_long = df.melt(
    id_vars="Gender",
    value_vars=["S-Learner", "T-Learner", "X-Learner"],
    var_name="Learner",
    value_name="CATE"
)

df_women = df_long[df_long["Gender"] == "Women"]
df_men   = df_long[df_long["Gender"] == "Men"]

# -------------------------------------------------------------------
# Publication-ready style
# -------------------------------------------------------------------
plt.style.use("default")
sns.set_theme(context="paper", style="whitegrid", font_scale=1.25)

fig = plt.figure(figsize=(18, 14))

# ============================================================
# PANEL A — Overall CATE Distributions (KDE)
# ============================================================
ax1 = fig.add_subplot(2, 2, 1)

for learner, color in zip(["S-Learner", "T-Learner", "X-Learner"], ["C0", "C1", "C2"]):
    sns.kdeplot(
        data=df_long[df_long["Learner"] == learner],
        x="CATE",
        fill=True, alpha=0.35, linewidth=1.6, label=learner, color=color, ax=ax1
    )

ax1.set_title("Panel A: Distribution of Estimated CATE")
ax1.set_xlabel("Estimated Treatment Effect")
ax1.set_ylabel("Density")
ax1.legend(frameon=True)
ax1.grid(alpha=0.3)

# ============================================================
# PANEL B — Gender × Learner KDE
# ============================================================
ax2 = fig.add_subplot(2, 2, 2)

# Women (solid)
sns.kdeplot(
    data=df_women, x="CATE", hue="Learner",
    fill=True, alpha=0.25, linewidth=1.7, ax=ax2
)

# Men (dashed)
sns.kdeplot(
    data=df_men, x="CATE", hue="Learner",
    fill=False, linewidth=1.7, linestyle="--", ax=ax2,
    legend=False
)

ax2.set_title("Panel B: Gender Heterogeneity in CATE")
ax2.set_xlabel("Estimated Treatment Effect")
ax2.set_ylabel("Density")
ax2.grid(alpha=0.3)

handles, labels = ax2.get_legend_handles_labels()
ax2.legend(handles, labels, title="Learner", frameon=True)

# ============================================================
# PANEL C — Histograms (S/T/X Learners)
# ============================================================
ax3 = fig.add_subplot(2, 2, 3)

bins = 35

ax3.hist(cate_S, bins=bins, alpha=0.45, color="C0", label="S-Learner")
ax3.hist(cate_T, bins=bins, alpha=0.45, color="C1", label="T-Learner")
ax3.hist(cate_X, bins=bins, alpha=0.45, color="C2", label="X-Learner")

ax3.set_title("Panel C: Histogram of CATE Estimates")
ax3.set_xlabel("Estimated Treatment Effect")
ax3.set_ylabel("Frequency")
ax3.legend(frameon=True)
ax3.grid(alpha=0.3)

# ============================================================
# PANEL D — Boxplots (Learner × Gender)
# ============================================================
ax4 = fig.add_subplot(2, 2, 4)

sns.boxplot(
    data=df_long, x="Learner", y="CATE", hue="Gender",
    palette="Set2", linewidth=1.2, fliersize=2.5, ax=ax4
)

ax4.set_title("Panel D: CATE by Learner Method and Gender")
ax4.set_xlabel("Learner")
ax4.set_ylabel("Estimated Treatment Effect")
ax4.legend(title="Gender", frameon=True)
ax4.grid(alpha=0.3)

plt.tight_layout()
plt.show()


### Non-linear DR

In [ ]:
# Get estimate (Doubly robust)
estimate = model.estimate_effect(
    identified_estimand=estimand,
    method_name='backdoor.econml.dr.DRLearner',
    target_units='ate',
    method_params={
        'init_params': {
            'model_propensity': LogisticRegression(),
            'model_regression': LGBMRegressor(n_estimators=1000, max_depth=10),
            'model_final': LGBMRegressor(n_estimators=500, max_depth=10),
        },
        'fit_params': {}
    })

In [ ]:
print("Mean CATE estimate:", estimate.cate_estimates.mean())

In [ ]:
# Compute predictions on new data
X_test = val_data.drop(['empl_06', 'select'], axis=1)
y_test = val_data['empl_06'].values

# Use the underlying EconML estimator
effect_pred = estimate.estimator.effect(X_test)

ate = np.mean(effect_pred)
std_eff = np.std(effect_pred)

# --- 3. Summarize treatment effects ---
ate = np.mean(effect_pred)
std_eff = np.std(effect_pred)

print(f"Average Treatment Effect (ATE): {ate:.4f}")
print(f"Standard Deviation of Estimated Effects: {std_eff:.4f}")

# Compute the error 
mean_absolute_error(y_test, effect_pred)

if 'dwomen' in X_test.columns:
    male_effect = np.mean(effect_pred[X_test['dwomen'] == 0])
    female_effect = np.mean(effect_pred[X_test['dwomen'] == 1])
    print(f"Average Effect (Men): {male_effect:.4f}")
    print(f"Average Effect (Women): {female_effect:.4f}")

In [ ]:
# Compute the error 
mean_absolute_percentage_error(effect_true, effect_pred)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

# --- Simple plot function ---
def plot_effect(effect_pred):
    """Plot the distribution of predicted treatment effects (CATEs)."""
    plt.figure(figsize=(7,5))
    sns.histplot(effect_pred, bins=30, kde=True, color="skyblue")
    plt.title("Distribution of Predicted Treatment Effects (CATEs)")
    plt.xlabel("Estimated Effect on Employment (ΔY)")
    plt.ylabel("Frequency")
    plt.show()

# --- Plot your results ---
plot_effect(effect_pred)


In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt
    
    # Plot treatment effect distribution by gender
plt.figure(figsize=(7, 5))
plt.hist(effect_pred[X_test['dwomen'] == 0], bins=30, alpha=0.6, label='Men')
plt.hist(effect_pred[X_test['dwomen'] == 1], bins=30, alpha=0.6, label='Women')
plt.title("Distribution of Estimated Treatment Effects (CATEs)")
plt.xlabel("Estimated Effect on Employment (ΔY)")
plt.ylabel("Frequency")
plt.legend()
plt.show()





# DMLEstimate the effect

In [ ]:
# Get estimate (DML)
estimate = model.estimate_effect(
    identified_estimand=estimand,
    method_name='backdoor.econml.dml.LinearDML',
    target_units='ate',
    method_params={
        'init_params': {
            'model_y': LGBMRegressor(n_estimators=500, max_depth=10),
            'model_t': LogisticRegression(),
            'discrete_treatment': True
        },
        'fit_params': {}
    })

In [ ]:
estimate.cate_estimates.mean()

### Predict on test data

In [ ]:
# Compute predictions
effect_pred = estimate.estimator.effect(val_data.drop(['empl_06', 'select'], axis=1))

# Get the true effect
effect_true = val_data['empl_06'].values

In [ ]:
# Compute the error 
mean_absolute_percentage_error(effect_true, effect_pred)

In [ ]:
plot_effect(
    effect_true=effect_true,
    effect_pred=effect_pred,
)

### DML with more folds

In [ ]:
# Estimate the effect
estimate = model.estimate_effect(
    identified_estimand=estimand,
    method_name='backdoor.econml.dml.LinearDML',
    target_units='ate',
    method_params={
        'init_params': {
            'model_y': LGBMRegressor(n_estimators=50, max_depth=10),
            'model_t': LogisticRegression(),
            'discrete_treatment': True,
            'cv': 4
        },
        'fit_params': {
        }
    })

In [ ]:
estimate.cate_estimates.mean()

In [ ]:
# Compute predictions
effect_pred = estimate.estimator.effect(val_data.drop(['empl_06', 'select'], axis=1))

# Get the true effect
effect_true = val_data['empl_06'].values

In [ ]:
# Compute the error 
mean_absolute_percentage_error(effect_true, effect_pred)

In [ ]:
plot_effect(
    effect_true=effect_true,
    effect_pred=effect_pred,
)

### DML with cross-validation

In [ ]:
# Define wrapped CV models
model_y = GridSearchCV(
    estimator=LGBMRegressor(),
    param_grid={
        'max_depth': [3, 10, 20, 100],
        'n_estimators': [10, 50, 100]
    }, cv=10, n_jobs=-1, scoring='neg_mean_squared_error'
)

model_t = GridSearchCV(
    estimator=LGBMClassifier(),
    param_grid={
        'max_depth': [3, 10, 20, 100],
        'n_estimators': [10, 50, 100]
    }, cv=10, n_jobs=-1, scoring='accuracy'
)

In [ ]:
# Estimate the effect
estimate = model.estimate_effect(
    identified_estimand=estimand,
    method_name='backdoor.econml.dml.LinearDML',
    target_units='ate',
    method_params={
        'init_params': {
            'model_y': model_y,
            'model_t': model_t,
            'discrete_treatment': True,
            'cv': 4
        },
        'fit_params': {
        }
    })

In [ ]:
estimate.cate_estimates.mean()

In [ ]:
# Compute predictions
effect_pred = estimate.estimator.effect(earnings_interaction_test.drop(['true_effect', 'took_a_course'], axis=1))

# Get the true effect
effect_true = earnings_interaction_test['true_effect'].values

In [ ]:
# Compute the error 
mean_absolute_percentage_error(effect_true, effect_pred)

In [ ]:
plot_effect(
    effect_true=effect_true,
    effect_pred=effect_pred,
)

## Causal Forests and more

In [ ]:
# Estimate the effect
estimate = model.estimate_effect(
    identified_estimand=estimand,
    method_name='backdoor.econml.dml.CausalForestDML',
    target_units='ate',
    method_params={
        'init_params': {
            'model_y': LGBMRegressor(n_estimators=50, max_depth=10),
            'model_t': LGBMClassifier(n_estimators=50, max_depth=10),
            'discrete_treatment': True,
            'cv': 4
        },
        'fit_params': {
        }
    }
)

In [ ]:
estimate.cate_estimates.mean()

In [ ]:
# Compute predictions
effect_pred = estimate.estimator.effect(earnings_interaction_test.drop(['true_effect', 'took_a_course'], axis=1))

# Get the true effect
effect_true = earnings_interaction_test['true_effect'].values

In [ ]:
# Compute the error 
mean_absolute_percentage_error(effect_true, effect_pred)

In [ ]:
plot_effect(
    effect_true=effect_true,
    effect_pred=effect_pred,
)

## Heterogenous Treatment Effects With Experimental Data

In [ ]:
# Read in the data
hillstrom_clean = pd.read_csv(r'./data/hillstrom_clean.csv')

# Read in labels mapping
with open(r'./data/hillstrom_clean_label_mapping.json', 'r') as f:
    hillstrom_labels_mapping = json.load(f)

In [ ]:
hillstrom_clean.head()

In [ ]:
# Drop redundant cols to avoid multicollinearity
hillstrom_clean = hillstrom_clean.drop(['zip_code__urban', 'channel__web'], axis=1)

### EDA

In [ ]:
# Display mapping
hillstrom_labels_mapping

In [ ]:
# P(visit)
hillstrom_clean.visit.mean()

In [ ]:
# P(conversion)
hillstrom_clean.conversion.mean()

In [ ]:
# Get sample size
sample_size = hillstrom_clean.shape[0]

In [ ]:
# Check how random is the treatment assignment

# Split data
hillstrom_X = hillstrom_clean.drop(['visit', 'conversion', 'spend', 'treatment'], axis=1)
hillstrom_Y = hillstrom_clean['spend']
hillstrom_T = hillstrom_clean['treatment']

In [ ]:
# P(T=t)
hillstrom_T.value_counts() / sample_size

In [ ]:
# Train-test split
X_train_eda, X_test_eda, T_train_eda, T_test_eda = train_test_split(hillstrom_X, hillstrom_T, test_size=.5)

In [ ]:
# Split quality
T_test_eda.value_counts() / T_test_eda.shape[0]

In [ ]:
# Fit the EDA model
lgbm_eda = LGBMClassifier()
lgbm_eda.fit(X_train_eda, T_train_eda)

In [ ]:
# Get predictions on the test
T_pred_eda = lgbm_eda.predict(X_test_eda)

# Check accuracy
acc_eda = accuracy_score(T_test_eda, T_pred_eda)
acc_eda

In [ ]:
# Generate random data
random_scores = []

test_eda_sample_size = T_test_eda.shape[0]

for i in range(10000):
    random_scores.append(
        (np.random.choice(
            [0, 1, 2], 
            test_eda_sample_size) == np.random.choice(
            [0, 1, 2], 
            test_eda_sample_size)).mean())
    
np.quantile(random_scores, .025), np.quantile(random_scores, .975)

In [ ]:
# Get 95% CIs
lower = np.quantile(random_scores, .025)
upper = np.quantile(random_scores, .975)
lower, upper

In [ ]:
# Plot radom vs accuracy
plt.figure(figsize=(10, 6))
plt.fill_betweenx(
    x1=lower, 
    x2=upper, 
    y=np.arange(0, 300),
    color=COLORS[0],
    alpha=.1,
    label='95% empirical CI'
)
plt.hist(random_scores, alpha=.7, color=COLORS[0], bins=100, label='Random models\n($n=10e3$)')
plt.axvline(acc_eda, color=COLORS[1], ls='--', label='Model accuracy')

plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))

plt.hist(
    hillstrom_clean[hillstrom_clean['treatment'] == 2]['spend'], 
    label=f'Men\'s', 
    color=COLORS[0],
    bins=100, 
    alpha=.5)

plt.hist(
    hillstrom_clean[hillstrom_clean['treatment'] == 1]['spend'], 
    label=f'Women\'s',
    color=COLORS[1],
    bins=100, 
    alpha=.5)

plt.legend()
plt.yscale('log')
plt.show()

In [ ]:
hillstrom_clean[hillstrom_clean['treatment'] == 1]['spend'].var()

In [ ]:
hillstrom_labels_mapping

### Modeling Hillstrom

In [ ]:
# No. of conversions
(hillstrom_Y[hillstrom_T > 0] > 0).sum()

In [ ]:
# Train test split
X_train, X_test, y_train, y_test, T_train, T_test = train_test_split(
    hillstrom_X,
    hillstrom_Y,
    hillstrom_T,
    test_size=.5
)

In [ ]:
# How many observations in train/test converted?
(y_train[T_train > 0] > 0).sum(), (y_test[T_test > 0] > 0).sum()

In [ ]:
def create_model(model_type, n_estimators=100, max_depth=10, learning_rate=.01):
    if model_type == 'regressor':
        return LGBMRegressor(
            n_estimators=n_estimators, 
            max_depth=max_depth, 
            learning_rate=learning_rate)
    elif model_type == 'classifier':
        return LGBMClassifier(
            n_estimators=n_estimators, 
            max_depth=max_depth, 
            learning_rate=learning_rate)
    else:
        raise NotImplementedError(f'Model type `{model_type}` not implemented.')

In [ ]:
# Models
s_learner = SLearner(
    overall_model=create_model('regressor')
)

x_learner = XLearner(
    models=[
        create_model('regressor'),
        create_model('regressor'),
        create_model('regressor'),
    ],
    cate_models=[
        create_model('regressor'),
        create_model('regressor'),
        create_model('regressor'),
    ]
)

t_learner = TLearner(
    models=[
        create_model('regressor'),
        create_model('regressor'),
        create_model('regressor'),
    ]
)

dml = LinearDML(
    model_y=create_model('regressor'),
    model_t=create_model('classifier'),
    discrete_treatment=True,
    cv=5
)

dr = DRLearner(
    model_propensity=LogisticRegression(),
    model_regression=create_model('regressor'),
    model_final=create_model('regressor'),
    cv=5,
)

cf = CausalForestDML(
    model_y=create_model('regressor'),
    model_t=create_model('classifier'),
    discrete_treatment=True,
    cv=5
)

In [ ]:
# Models
models = {
    'SLearner': s_learner,
    'TLearner': t_learner,
    'XLearner': x_learner,
    'DRLearner': dr,
    'LinearDML': dml,
    'CausalForestDML': cf
} 

In [ ]:
# Fit the estimator
for model_name, model in models.items():
    start = time.time()
    print(f'Fitting {model_name}')
    model.fit(
        Y=y_train,
        T=T_train,
        X=X_train
    )
    stop = time.time()
    
    print(f'{model_name} fitted in {stop - start:0.4f} seconds.\n')

In [ ]:
# Compute effects
effects_train = {
    'treatment_1': {},
    'treatment_2': {}
}

effects_test = {
    'treatment_1': {},
    'treatment_2': {}
}


for treatment in [1, 2]:
    for model_name, model in tqdm(models.items()):
        
        # Compute effects on train
        effects_local_train = models[model_name].effect(X_train.values, T0=0, T1=treatment)
        effects_train[f'treatment_{treatment}'][model_name] = effects_local_train
        
        # Compute effects on test
        effects_local_test = models[model_name].effect(X_test.values, T0=0, T1=treatment)
        effects_test[f'treatment_{treatment}'][model_name] = effects_local_test


#### Uplift by decile

In [ ]:
def get_uplift_by_decile(uplifts, t_true, t_pred, y_true):
    
    # Encapsulate the data & sort according to uplifts
    all_data = pd.DataFrame(
        dict(
            uplifts=uplifts, 
            y_true=y_true, 
            t_true=t_true)
    ).query(f't_true==0 | t_true=={t_pred}').sort_values('uplifts')
    
    # Partition into deciles
    all_data['deciles'] = pd.qcut(all_data['uplifts'], q=10, labels=np.arange(10), duplicates='raise')
    
    # Get mean responses within deciles
    mean_decile_resp = all_data.groupby(['deciles', 't_true']).mean()
    
    # Compute true decile uplift
    mean_decile_resp['true_uplift'] = mean_decile_resp['y_true'] * np.array([-1, 1]*10)
    true_uplift = mean_decile_resp.groupby(level=[0]).sum()['true_uplift']  
    
    return true_uplift[::-1]

In [ ]:
plt.figure(figsize=(40, 30))

i = 1

for model_name in models.keys():
    
    uplifts_by_decile = {
        'treatment_1': {},
        'treatment_2': {}
    }
    
    global_min = np.inf
    global_max = -np.inf
    
    for treatment in ['treatment_1', 'treatment_2']:

        uplift_by_decile_train = get_uplift_by_decile(
            uplifts=effects_train[treatment][model_name], 
            t_true=T_train,
            t_pred=int(treatment.split('_')[-1]),
            y_true=y_train
        )

        uplift_by_decile_test = get_uplift_by_decile(
            uplifts=effects_test[treatment][model_name], 
            t_true=T_test,
            t_pred=int(treatment.split('_')[-1]),
            y_true=y_test
        )
            
        plt.subplot(6, 4, i)
        plt.bar(np.arange(10), uplift_by_decile_train, color=COLORS[0])
        plt.title(f'{model_name} {treatment} - Train')
        
        plt.subplot(6, 4, i + 1)
        plt.bar(np.arange(10), uplift_by_decile_test, color=COLORS[1])
        plt.title(f'{model_name} {treatment} - Test')
        
        i += 2
        
plt.show()

#### Expected response

As introduced by [Zhao et al., 2017](https://arxiv.org/pdf/1705.08492.pdf)

Formula:

$$\Large Z = \sum_{t=0}^K \frac{1}{P(T=t)}y \mathbb{I}_{h(x)=t}\mathbb{I}_{T=t}$$

<br>


* $K$ is a number of treatment levels

* $\mathbb{I}$ is an indicator function

* $h(x)$ is the treatment recommended by the model (treatment leading to the highest uplift)

In [ ]:
def get_effects_argmax(effects_arrays, return_matrix=False):
    """Returns argmax for each row of predicted effects for the arbitrary no. of treatments.
    
    :param effects_arrays: A list of arrays for K treatments, where K>=1 (without control null effects)
    :type effects_arrays: list of np.arrays
    
    :param return_matrix: Determines if the function returns a matrix of all effects 
        (with added null effect for control)
    :type return_matrix: bool

    ...
    :return: A stacked matrix of all effects with added column for control effects (which is always 0)
    :rtype: np.array
    """
    
    n_rows = effects_arrays[0].shape[0]
    null_effect_array = np.zeros(n_rows)
    stacked = np.stack([null_effect_array] + effects_arrays).T
    
    if return_matrix:
        return np.argmax(stacked, axis=1), stacked
    
    return np.argmax(stacked, axis=1)


def get_expected_response(y_true, t_true, effects_argmax):
    """Computes the average expected response for an uplift model according to the formula
        proposed by: 
        Zhao, Y., Fang, X., & Simchi-Levi, D. (2017). Uplift Modeling with Multiple Treatments and General Response Types. 
        Proceedings of the 2017 SIAM International Conference on Data Mining, 588-596. 
        Society for Industrial and Applied Mathematics.   
    """
    
    proba_t = pd.Series(t_true).value_counts() / np.array(t_true).shape[0]
    treatments = proba_t.index.values
    
    z_vals = 0
    
    for treatment in treatments:
        h_indicator = effects_argmax == treatment
        t_indicator = t_true == treatment
        t_proba_local = proba_t[treatment]
        
        z_vals += (1/t_proba_local) * y_true * h_indicator * t_indicator
    
    return z_vals.mean()

In [ ]:
# Compute expected response
print('Expecetd response on train:\n')
for model_name in models:
    effects_argmax = get_effects_argmax(
        [
            effects_train['treatment_1'][model_name],
            effects_train['treatment_2'][model_name]
        ]
    )
    
    expected_response = get_expected_response(
        y_true=y_train, 
        t_true=T_train, 
        effects_argmax=effects_argmax
    )
    
    print(f'{model_name}: {expected_response}')
    
print('\n' + '-'*30)
    
print('Expected response on test:\n')
for model_name in models:
    effects_argmax = get_effects_argmax(
        [
            effects_test['treatment_1'][model_name],
            effects_test['treatment_2'][model_name]
        ]
    )
    
    expected_response = get_expected_response(
        y_true=y_test, 
        t_true=T_test, 
        effects_argmax=effects_argmax
    )
    
    print(f'{model_name}: {expected_response}')

In [ ]:
# Outcome in the whole dataset
hillstrom_clean.groupby('treatment')['spend'].mean()

#### Confidence intervals

In [ ]:
models['LinearDML'].effect_interval(X=X_test, T0=0, T1=1)

In [ ]:
models['LinearDML'].effect_interval(X=X_test.iloc[0:1, :], T0=0, T1=1)

In [ ]:
# CIs (DML)
ints = np.stack(models['LinearDML'].effect_interval(X=X_test, T0=0, T1=1, alpha=.05)).T

# What % of effects contains zero?
(np.sign(ints[:, 0]) == np.sign(ints[:, 1])).sum() / ints.shape[0]